# Q5 Kaggle — Twitter Bot vs. Genuine User Classification

**Competition:** CS610 Assignment 1 Question 5 (2026)
**Evaluation metric:** AUC (Area Under ROC Curve)
**Submission format:** `index,target` — where `target` is a probability in `[0, 1]`

This notebook walks through baseline → LightGBM → cross-validation → text features → temporal features → Optuna tuning. Each section adds one specific change and validates it through CV before submitting.


## 1. Setup

In [ ]:
from pathlib import Path
import warnings, logging

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from scipy import sparse

warnings.filterwarnings("ignore", category=UserWarning)
RANDOM_STATE = 2025


In [ ]:
HERE = Path.cwd()
REPO = HERE.parent if HERE.name == "q5_kaggle" else HERE
DATA_DIR = REPO / "Assignment1" / "cs-610-assignment-1-question-5-2026"
SUB_DIR = (HERE if HERE.name == "q5_kaggle" else HERE / "q5_kaggle") / "submissions"
SUB_DIR.mkdir(exist_ok=True, parents=True)
print("Data dir :", DATA_DIR)
print("Sub  dir :", SUB_DIR)


## 2. Load and inspect

In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv")
test  = pd.read_csv(DATA_DIR / "test.csv")
print(f"train: {train.shape}")
print(f"test : {test.shape}")
train.head()


In [ ]:
print(train["target"].value_counts())
print(f"\nbot rate: {train['target'].mean():.3f}")


In [ ]:
pd.DataFrame({
    "dtype":       train.dtypes,
    "missing":     train.isna().sum(),
    "pct_missing": (train.isna().mean() * 100).round(2),
    "nunique":     train.nunique(),
})


- Class imbalance is mild (33% bots). AUC is robust to imbalance.
- `description` and `lang` ~20% missing — impute + presence flags.
- `screen_name` is unique per row — extract length/digit-fraction patterns.
- `profile_background_image_url` has 20 unique values — Twitter defaults.
- `id` is unique — drop. `created_at` — handled in §9.


## 3. Feature engineering

The raw columns aren't directly usable. We tame heavy-tailed counts with `log1p`, convert booleans to int, bucket high-cardinality `lang` to top-8 + `OTHER`, extract cheap text features, and add ratio features.


In [ ]:
TOP_LANGS = train["lang"].value_counts().head(8).index.tolist()
print("Top 8 langs:", TOP_LANGS)


In [ ]:
def featurize(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)
    counts = ["favourites_count", "followers_count", "friends_count",
              "statuses_count", "average_tweets_per_day", "account_age_days"]
    for c in counts:
        out[c] = df[c].astype(float)
    for c in ["favourites_count", "followers_count", "friends_count", "statuses_count"]:
        out[f"log_{c}"] = np.log1p(out[c])
    for c in ["default_profile", "default_profile_image", "geo_enabled", "verified"]:
        out[c] = df[c].astype(int)
    out["has_description"] = df["description"].notna().astype(int)
    out["has_location"]    = (df["location"].notna() & (df["location"] != "unknown")).astype(int)
    out["has_url_bg"]      = df["profile_background_image_url"].notna().astype(int)
    out["desc_len"]  = df["description"].fillna("").str.len()
    out["sn_len"]    = df["screen_name"].fillna("").str.len()
    out["sn_digits"] = (df["screen_name"].fillna("").str.count(r"\d") / out["sn_len"].clip(lower=1))
    lang = df["lang"].fillna("MISSING")
    out["lang"] = lang.where(lang.isin(TOP_LANGS), "OTHER")
    out["followers_per_friend"] = df["followers_count"] / df["friends_count"].clip(lower=1)
    out["statuses_per_day"]     = df["statuses_count"]  / df["account_age_days"].clip(lower=1)
    return out

X_train_full = featurize(train)
y_train      = train["target"].values
X_test       = featurize(test)
print(f"feature matrix: train {X_train_full.shape}, test {X_test.shape}")


## 4. Train/validation split

In [ ]:
Xa, Xb, ya, yb = train_test_split(
    X_train_full, y_train,
    test_size=0.2, stratify=y_train, random_state=RANDOM_STATE,
)
print(f"train: {Xa.shape}  val: {Xb.shape}")


## 5. LogReg + Random Forest baseline

Two preprocessors: LogReg needs scaled numerics + one-hot lang; RF only needs one-hot lang.


In [ ]:
numeric_cols     = [c for c in X_train_full.columns if c != "lang"]
categorical_cols = ["lang"]

preproc_lr = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                      ("scale",  StandardScaler())]), numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
])

preproc_rf = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
])

models = {
    "logreg": Pipeline([("preproc", preproc_lr),
                        ("clf", LogisticRegression(max_iter=2000, C=1.0,
                                                   random_state=RANDOM_STATE))]),
    "rf": Pipeline([("preproc", preproc_rf),
                    ("clf", RandomForestClassifier(n_estimators=300, min_samples_leaf=2,
                                                   n_jobs=-1, random_state=RANDOM_STATE))]),
}

scores = {}
for name, pipe in models.items():
    pipe.fit(Xa, ya)
    yb_prob = pipe.predict_proba(Xb)[:, 1]
    yb_pred = (yb_prob >= 0.5).astype(int)
    auc = roc_auc_score(yb, yb_prob); f1 = f1_score(yb, yb_pred)
    scores[name] = {"auc": auc, "f1": f1}
    print(f"=== {name.upper()} ===   AUC: {auc:.4f}   F1: {f1:.4f}")


In [ ]:
winner = max(scores, key=lambda k: scores[k]["auc"])
best = models[winner]
best.fit(X_train_full, y_train)
test_prob = best.predict_proba(X_test)[:, 1]
sub = pd.DataFrame({"index": test["index"].values, "target": test_prob})
sub.to_csv(SUB_DIR / f"baseline_{winner}.csv", index=False, float_format="%.6f")
print(f"Wrote: {SUB_DIR / f'baseline_{winner}.csv'}")


## 6. Upgrade to LightGBM

GBM trains trees sequentially — each new tree fits the residual errors of the previous trees. Two LightGBM features that help: native categorical handling (`lang` as `category`), and early stopping (library picks the right number of trees from `eval_set`).


In [ ]:
import lightgbm as lgb

def to_lgb_input(X):
    out = X.copy()
    out["lang"] = out["lang"].astype("category")
    return out

X_lgb_full = to_lgb_input(X_train_full)
X_lgb_test = to_lgb_input(X_test)
X_lgb_test["lang"] = pd.Categorical(
    X_lgb_test["lang"], categories=X_lgb_full["lang"].cat.categories
)

Xa_lgb, Xb_lgb, ya_lgb, yb_lgb = train_test_split(
    X_lgb_full, y_train, test_size=0.2, stratify=y_train, random_state=RANDOM_STATE,
)

lgbm = lgb.LGBMClassifier(
    n_estimators=2000, learning_rate=0.05, num_leaves=63,
    min_data_in_leaf=20, feature_fraction=0.9, bagging_fraction=0.9,
    bagging_freq=5, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1,
)
lgbm.fit(Xa_lgb, ya_lgb, eval_set=[(Xb_lgb, yb_lgb)], eval_metric="auc",
         callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])

yb_prob_lgb = lgbm.predict_proba(Xb_lgb)[:, 1]
auc_lgb = roc_auc_score(yb_lgb, yb_prob_lgb)
print(f"LightGBM holdout AUC : {auc_lgb:.4f}")
print(f"Random Forest        : {scores['rf']['auc']:.4f}")
print(f"Lift                 : {auc_lgb - scores['rf']['auc']:+.4f}")
print(f"Best iter            : {lgbm.best_iteration_}")


In [ ]:
lgbm_final = lgb.LGBMClassifier(
    n_estimators=lgbm.best_iteration_, learning_rate=0.05, num_leaves=63,
    min_data_in_leaf=20, feature_fraction=0.9, bagging_fraction=0.9, bagging_freq=5,
    random_state=RANDOM_STATE, n_jobs=-1, verbose=-1,
)
lgbm_final.fit(X_lgb_full, y_train)
test_prob_lgb = lgbm_final.predict_proba(X_lgb_test)[:, 1]
sub = pd.DataFrame({"index": test["index"].values, "target": test_prob_lgb})
sub.to_csv(SUB_DIR / "lgbm_v1.csv", index=False, float_format="%.6f")
print(f"Wrote: {SUB_DIR/'lgbm_v1.csv'}")


**Public LB:** baseline_rf.csv → 0.93470, lgbm_v1.csv → 0.93553

## 7. Stratified 5-fold cross-validation

Single 80/20 holdouts have variance — we need a stable benchmark. 5-fold CV gives us mean ± std (the std is our noise floor) and OOF predictions (one ranking across all training rows, useful later for ensembling).


In [ ]:
def lgbm_cv(X, y, n_splits=5, params=None, verbose=True):
    """5-fold CV with LightGBM + early stopping. Works on DataFrame or sparse matrix."""
    p = dict(
        n_estimators=2000, learning_rate=0.05, num_leaves=63,
        min_data_in_leaf=20, feature_fraction=0.9, bagging_fraction=0.9,
        bagging_freq=5, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1,
    )
    if params: p.update(params)

    def take(M, idx):
        return M.iloc[idx] if hasattr(M, "iloc") else M[idx]

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    aucs, iters = [], []
    n_rows = X.shape[0]
    oof = np.zeros(n_rows)

    for fold, (tr, va) in enumerate(skf.split(np.zeros(n_rows), y)):
        clf = lgb.LGBMClassifier(**p)
        clf.fit(take(X, tr), y[tr],
                eval_set=[(take(X, va), y[va])], eval_metric="auc",
                callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
        prob = clf.predict_proba(take(X, va))[:, 1]
        oof[va] = prob
        auc = roc_auc_score(y[va], prob)
        aucs.append(auc); iters.append(clf.best_iteration_)
        if verbose:
            print(f"Fold {fold+1}: AUC={auc:.4f}  best_iter={clf.best_iteration_}")

    aucs = np.array(aucs)
    if verbose:
        print(f"\nMean fold AUC : {aucs.mean():.4f} ± {aucs.std():.4f}")
        print(f"OOF AUC       : {roc_auc_score(y, oof):.4f}")
        print(f"Mean best_iter: {int(np.mean(iters))}")
    return {"aucs": aucs, "iters": iters, "oof": oof}

cv_baseline = lgbm_cv(X_lgb_full, y_train)


**Benchmark:** OOF AUC ≈ 0.9411, std ≈ 0.0035.

## 8. Description text features (TF-IDF)

Bots often have empty/templated bios. TF-IDF: term frequency × inverse document frequency. Common terms get low weight, distinctive terms get high. Result is sparse.

Critical: fit TF-IDF on TRAIN only, then transform both. Fitting on test would leak test vocabulary.


In [ ]:
text_train = train["description"].fillna("")
text_test  = test["description"].fillna("")

tfidf = TfidfVectorizer(
    max_features=500, ngram_range=(1, 2),
    min_df=5, max_df=0.95, lowercase=True,
    sublinear_tf=True, strip_accents="unicode",
)
Xt_train = tfidf.fit_transform(text_train)
Xt_test  = tfidf.transform(text_test)
print(f"TF-IDF: {Xt_train.shape}")


In [ ]:
oh = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
lang_train_oh = oh.fit_transform(X_train_full[["lang"]])
lang_test_oh  = oh.transform(X_test[["lang"]])

num_train = sparse.csr_matrix(X_train_full[numeric_cols].fillna(0).values)
num_test  = sparse.csr_matrix(X_test[numeric_cols].fillna(0).values)

X_combined_train = sparse.hstack([num_train, lang_train_oh, Xt_train]).tocsr()
X_combined_test  = sparse.hstack([num_test,  lang_test_oh,  Xt_test ]).tocsr()
print(f"Combined: {X_combined_train.shape}")


In [ ]:
cv_text = lgbm_cv(X_combined_train, y_train)
print(f"\nNo text  OOF: {roc_auc_score(y_train, cv_baseline['oof']):.4f}")
print(f"With text OOF: {roc_auc_score(y_train, cv_text['oof']):.4f}")


In [ ]:
clf_v2 = lgb.LGBMClassifier(
    n_estimators=int(np.mean(cv_text["iters"])),
    learning_rate=0.05, num_leaves=63, min_data_in_leaf=20,
    feature_fraction=0.9, bagging_fraction=0.9, bagging_freq=5,
    random_state=RANDOM_STATE, n_jobs=-1, verbose=-1,
)
clf_v2.fit(X_combined_train, y_train)
test_prob = clf_v2.predict_proba(X_combined_test)[:, 1]
sub = pd.DataFrame({"index": test["index"].values, "target": test_prob})
sub.to_csv(SUB_DIR / "lgbm_v2_text.csv", index=False, float_format="%.6f")
print(f"Wrote: {SUB_DIR/'lgbm_v2_text.csv'}")


**Public LB:** lgbm_v2_text.csv → 0.93919

## 9. Temporal features from `created_at`

Year, month, day-of-week, hour-of-day, plus cyclical sin/cos encodings of hour and DoW. The cyclical pair lets a linear model represent "23:00 is close to 00:00" — for tree models the raw integer is fine, but we include both for ensemble robustness later.


In [ ]:
def add_temporal(df: pd.DataFrame, X_dense: pd.DataFrame) -> pd.DataFrame:
    out = X_dense.copy()
    ct = pd.to_datetime(df["created_at"])
    out["created_year"]  = ct.dt.year.astype(float)
    out["created_month"] = ct.dt.month.astype(float)
    out["created_dow"]   = ct.dt.dayofweek.astype(float)
    out["created_hour"]  = ct.dt.hour.astype(float)
    out["hour_sin"] = np.sin(2*np.pi*out["created_hour"]/24)
    out["hour_cos"] = np.cos(2*np.pi*out["created_hour"]/24)
    out["dow_sin"]  = np.sin(2*np.pi*out["created_dow"]/7)
    out["dow_cos"]  = np.cos(2*np.pi*out["created_dow"]/7)
    return out

X_v3_train = add_temporal(train, X_train_full)
X_v3_test  = add_temporal(test,  X_test)
new_cols = [c for c in X_v3_train.columns if c not in X_train_full.columns]
print(f"Added temporal cols: {new_cols}")


In [ ]:
numeric_cols_v3 = [c for c in X_v3_train.columns if c != "lang"]

num_train_v3 = sparse.csr_matrix(X_v3_train[numeric_cols_v3].fillna(0).values)
num_test_v3  = sparse.csr_matrix(X_v3_test[numeric_cols_v3].fillna(0).values)

X_combined_train_v3 = sparse.hstack([num_train_v3, lang_train_oh, Xt_train]).tocsr()
X_combined_test_v3  = sparse.hstack([num_test_v3,  lang_test_oh,  Xt_test ]).tocsr()
print(f"Combined v3: {X_combined_train_v3.shape}")

cv_temporal = lgbm_cv(X_combined_train_v3, y_train)


In [ ]:
clf_v3 = lgb.LGBMClassifier(
    n_estimators=int(np.mean(cv_temporal["iters"])),
    learning_rate=0.05, num_leaves=63, min_data_in_leaf=20,
    feature_fraction=0.9, bagging_fraction=0.9, bagging_freq=5,
    random_state=RANDOM_STATE, n_jobs=-1, verbose=-1,
)
clf_v3.fit(X_combined_train_v3, y_train)
test_prob_v3 = clf_v3.predict_proba(X_combined_test_v3)[:, 1]
sub = pd.DataFrame({"index": test["index"].values, "target": test_prob_v3})
sub.to_csv(SUB_DIR / "lgbm_v3_temporal.csv", index=False, float_format="%.6f")
print(f"Wrote: {SUB_DIR/'lgbm_v3_temporal.csv'}")


**Public LB:** lgbm_v3_temporal.csv → 0.93850 (within noise of v2's 0.93919)

## 10. Hyperparameter tuning with Optuna

We've stopped getting easy wins from features. Time to tune the model. Optuna uses **Bayesian optimization** — it doesn't randomly try hyperparameters, it learns from past trials to predict where good ones are likely to be, then samples there. Specifically TPE (Tree-structured Parzen Estimator) — fits two models, one for "good trial" parameter distributions and one for "bad trial," and picks the next trial to maximize their ratio.

Practical workflow:
1. **Define the search space.** What's the range for each hyperparameter? Use `log=True` for parameters that span orders of magnitude (learning rate, regularization).
2. **Define the objective.** A function that takes a `trial` object, asks it for parameter values, runs CV, returns the metric to maximize.
3. **Run** `study.optimize(objective, n_trials=N)`. Optuna handles everything else.

Set `n_trials=50` for a meaningful search (~15 minutes on this dataset). Bump to 100+ for the final run before submission.


In [ ]:
import optuna
logging.getLogger("optuna").setLevel(logging.WARNING)  # quiet the per-trial logs

def objective(trial):
    params = dict(
        learning_rate    = trial.suggest_float("learning_rate", 0.02, 0.15, log=True),
        num_leaves       = trial.suggest_int("num_leaves", 15, 200),
        min_data_in_leaf = trial.suggest_int("min_data_in_leaf", 5, 80),
        feature_fraction = trial.suggest_float("feature_fraction", 0.5, 1.0),
        bagging_fraction = trial.suggest_float("bagging_fraction", 0.5, 1.0),
        bagging_freq     = trial.suggest_int("bagging_freq", 1, 10),
        lambda_l1        = trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
        lambda_l2        = trial.suggest_float("lambda_l2", 1e-8, 10.0, log=True),
    )
    cv = lgbm_cv(X_combined_train_v3, y_train, params=params, verbose=False)
    return roc_auc_score(y_train, cv["oof"])

sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE)
study = optuna.create_study(direction="maximize", sampler=sampler,
                            study_name="lgbm_v3_tuning")

# Set N_TRIALS=50 for a real run (~15 min). Bump to 100+ before final submission.
N_TRIALS = 50
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f"\nBest OOF AUC: {study.best_value:.4f}")
print(f"Baseline (default params): 0.9435")
print(f"Lift: {study.best_value - 0.9435:+.4f}")
print(f"\nBest params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")


### Inspecting the search

Optuna keeps the history. Two views worth looking at:

- **Optimization history** — does AUC plateau, or is there room for more trials?
- **Parameter importances** — which knobs matter most? Often surprising.


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# History
trials_df = study.trials_dataframe()
axes[0].plot(trials_df["number"], trials_df["value"], "o-", alpha=0.5, label="trial")
running_best = trials_df["value"].cummax()
axes[0].plot(trials_df["number"], running_best, "-", linewidth=2, color="red", label="best so far")
axes[0].set_xlabel("trial number"); axes[0].set_ylabel("OOF AUC")
axes[0].set_title("Optimization progress"); axes[0].legend(); axes[0].grid(alpha=0.3)

# Param importance
try:
    importances = optuna.importance.get_param_importances(study)
    pd.Series(importances).sort_values().plot.barh(ax=axes[1])
    axes[1].set_title("Hyperparameter importance"); axes[1].set_xlabel("importance")
except Exception as e:
    axes[1].text(0.5, 0.5, f"Importances need >2 trials\n{e}", ha="center")
plt.tight_layout(); plt.show()


In [ ]:
# Refit on full train with best params, predict test, write submission
best_params = dict(study.best_params)
# Re-discover n_estimators via early stopping on the full feature set + a holdout slice
Xa3, Xb3, ya3, yb3 = train_test_split(
    np.arange(X_combined_train_v3.shape[0]), y_train,
    test_size=0.2, stratify=y_train, random_state=RANDOM_STATE,
)
es_clf = lgb.LGBMClassifier(
    n_estimators=3000, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1,
    **best_params,
)
es_clf.fit(
    X_combined_train_v3[Xa3], y_train[Xa3],
    eval_set=[(X_combined_train_v3[Xb3], y_train[Xb3])],
    eval_metric="auc",
    callbacks=[lgb.early_stopping(80), lgb.log_evaluation(0)],
)
final_iters = es_clf.best_iteration_
print(f"Tuned best_iter: {final_iters}")

clf_v4 = lgb.LGBMClassifier(
    n_estimators=final_iters,
    random_state=RANDOM_STATE, n_jobs=-1, verbose=-1,
    **best_params,
)
clf_v4.fit(X_combined_train_v3, y_train)
test_prob_v4 = clf_v4.predict_proba(X_combined_test_v3)[:, 1]
sub = pd.DataFrame({"index": test["index"].values, "target": test_prob_v4})
sub.to_csv(SUB_DIR / "lgbm_v4_tuned.csv", index=False, float_format="%.6f")
print(f"Wrote: {SUB_DIR/'lgbm_v4_tuned.csv'}")


## What's next

| Submission | OOF AUC | Public LB |
|---|---|---|
| `baseline_rf.csv` | n/a | 0.93470 |
| `lgbm_v1.csv` | n/a | 0.93553 |
| `lgbm_v2_text.csv` | 0.9429 | 0.93919 |
| `lgbm_v3_temporal.csv` | 0.9435 | 0.93850 |
| `lgbm_v4_tuned.csv` | TBD (target ~0.945+) | TBD |

Submit `lgbm_v4_tuned.csv` and tell me the public number. Then:

1. **Ensemble** — train XGBoost on the same features, average its probs with LightGBM. Diversity helps. ~+0.001–0.003 reliably.
2. **More text** — character n-grams, screen_name TF-IDF, larger vocabulary. Each one a small experiment.
3. **Final pick.** At submission deadline, choose 2 from CV-best + diverse fallback.
